## Decision Trees with scikit-learn
Decision trees are a fundamental concept in machine learning, used for both classification and regression tasks. They are interpretable, handle non-linearity, and form the basis for ensemble methods like random forests and gradient boosting. In this notebook, we will explore how to use decision trees with scikit-learn, including how to control tree complexity, handle imbalanced data, and interpret tree decisions.

### Importing Necessary Libraries
We start by importing the necessary libraries. We will use `numpy` for numerical computations, `sklearn.tree` for decision trees, and `sklearn.model_selection` for splitting our data into training and testing sets.

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris, make_classification
from sklearn.metrics import accuracy_score, mean_squared_error
import matplotlib.pyplot as plt

### Setting the Random Seed
For reproducibility, we set the random seed.

In [ ]:
np.random.seed(42)

### Classification Tree
Decision trees can be used for classification tasks. They work by recursively splitting the data to create hierarchical decision rules, with each split maximizing information gain (reducing impurity).

In [ ]:
iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("=== Decision Tree Classification ===")
print(f"Dataset: {len(X)} samples, {X.shape[1]} features")
print(f"Classes: {np.unique(y)} ({len(np.unique(y))} classes)")

The output shows the number of samples and features in our dataset, as well as the classes present. This information is crucial for understanding the structure of our data and for selecting appropriate parameters for our decision tree classifier.

### Training a Decision Tree Classifier
We train a decision tree classifier using the training data. The classifier splits the data based on the Gini impurity criterion by default, which measures the class mixing in the data.

In [ ]:
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)

The decision tree is now trained and can be used to make predictions on new, unseen data.

### Making Predictions
We use the trained decision tree to make predictions on the test data.

In [ ]:
y_pred = tree.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nAccuracy: {accuracy:.4f}")

The accuracy of the model is printed out. This gives us an idea of how well the model is performing on unseen data.

### Feature Importance
Decision trees can provide feature importance scores, which indicate how much each feature contributes to the decisions made by the tree.

In [ ]:
feature_importance = tree.feature_importances_
top_features = np.argsort(feature_importance)[::-1][:3]
print(f"\nTop 3 most important features:")
for feat_idx in top_features:
    print(f"  {iris.feature_names[feat_idx]}: {feature_importance[feat_idx]:.4f}")

The feature importance scores are useful for understanding which features are driving the predictions of the model.

### Tree Depth and Number of Leaves
The depth of the tree and the number of leaves can give us insight into the complexity of the model.

In [ ]:
print(f"\nTree depth: {tree.get_depth()}")
print(f"Number of leaves: {tree.get_n_leaves()}")

A deeper tree with more leaves may be more prone to overfitting, while a shallower tree with fewer leaves may be more prone to underfitting.

### Controlling Tree Complexity
To prevent overfitting, we can control the complexity of the tree by limiting its depth or the number of leaves.

In [ ]:
depths = [1, 3, 5, 10, 20]
train_acc = []
test_acc = []

for depth in depths:
    tree_depth = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree_depth.fit(X_train, y_train)
    train_acc.append(tree_depth.score(X_train, y_train))
    test_acc.append(tree_depth.score(X_test, y_test))

for depth, train, test in zip(depths, train_acc, test_acc):
    print(f"max_depth={depth}: Train={train:.4f}, Test={test:.4f}", end="")
    if train > test + 0.05:
        print(" → overfitting")
    else:
        print()

By controlling the complexity of the tree, we can find a balance between fitting the training data well and generalizing to new data.

### Regression Tree
Decision trees can also be used for regression tasks, where the goal is to predict a continuous target variable.

In [ ]:
from sklearn.datasets import make_regression
X_reg, y_reg = make_regression(n_samples=100, n_features=5, noise=10, random_state=42)
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

tree_reg = DecisionTreeRegressor(max_depth=5, random_state=42)
tree_reg.fit(X_reg_train, y_reg_train)

y_reg_pred = tree_reg.predict(X_reg_test)
mse = mean_squared_error(y_reg_test, y_reg_pred)
rmse = np.sqrt(mse)

print(f"RMSE: {rmse:.4f}")
print(f"R² score: {tree_reg.score(X_reg_test, y_reg_test):.4f}")

The regression tree predicts the target variable by creating average predictions per leaf.

### Split Criterion: Gini vs Entropy
The split criterion used by the decision tree can affect its performance. The Gini impurity and entropy are two common criteria used.

In [ ]:
tree_gini = DecisionTreeClassifier(criterion='gini', random_state=42)
tree_gini.fit(X_train, y_train)
acc_gini = tree_gini.score(X_test, y_test)

tree_entropy = DecisionTreeClassifier(criterion='entropy', random_state=42)
tree_entropy.fit(X_train, y_train)
acc_entropy = tree_entropy.score(X_test, y_test)

print(f"Gini criterion accuracy: {acc_gini:.4f}")
print(f"Entropy criterion accuracy: {acc_entropy:.4f}")

The choice of split criterion can depend on the specific problem and dataset.

### Handling Imbalanced Data
Decision trees can be sensitive to imbalanced data, where one class has a significantly larger number of instances than the others.

In [ ]:
X_imbal, y_imbal = make_classification(n_samples=1000, n_features=5, n_classes=2, weights=[0.9, 0.1], random_state=42)

tree_standard = DecisionTreeClassifier(random_state=42)
tree_standard.fit(X_imbal, y_imbal)
pred_standard = tree_standard.predict(X_imbal)

tree_balanced = DecisionTreeClassifier(class_weight='balanced', random_state=42)
tree_balanced.fit(X_imbal, y_imbal)
pred_balanced = tree_balanced.predict(X_imbal)

print(f"Standard tree: predicts class 1 {(pred_standard==1).sum()} times")
print(f"Balanced tree: predicts class 1 {(pred_balanced==1).sum()} times")

By using class weights, we can improve the performance of the decision tree on the minority class.

### Feature Scaling Not Required
Decision trees do not require feature scaling, as they do not use distance metrics.

In [ ]:
X_unscaled = np.array([[1000, 0.5], [2000, 1.0], [3000, 1.5]]) * 100
X_unscaled = X_unscaled.astype(float)
y_simple = np.array([0, 1, 1])

tree_unscaled = DecisionTreeClassifier(random_state=42)
tree_unscaled.fit(X_unscaled, y_simple)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_unscaled)
tree_scaled = DecisionTreeClassifier(random_state=42)
tree_scaled.fit(X_scaled, y_simple)

print(f"Unscaled accuracy: {tree_unscaled.score(X_unscaled, y_simple):.4f}")
print(f"Scaled accuracy: {tree_scaled.score(X_scaled, y_simple):.4f}")

The accuracy of the decision tree is the same whether the features are scaled or not.

### Tree Interpretability: Decision Paths
Decision trees are highly interpretable, as each path through the tree represents a decision rule.

In [ ]:
decision_path = tree.decision_path(X_test[:1])
leaf_id = tree.apply(X_test[:1])

print(f"Sample #0 features: {X_test[0]}")
print(f"Predicted class: {y_pred[0]}")
print(f"Prediction path complexity: {decision_path.nnz[0]} nodes visited")

By examining the decision path, we can understand how the decision tree arrived at its prediction.

### Practical ML Scenarios
Decision trees have many practical applications in machine learning.

In [ ]:
tree_deep = DecisionTreeClassifier(random_state=42)  # No depth limit
tree_deep.fit(X_train, y_train)
print(f"  Deep tree (unpruned):")
print(f"    Train accuracy: {tree_deep.score(X_train, y_train):.4f}")
print(f"    Test accuracy: {tree_deep.score(X_test, y_test):.4f}")

tree_shallow = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_shallow.fit(X_train, y_train)
print(f"  Shallow tree (max_depth=3):")
print(f"    Train accuracy: {tree_shallow.score(X_train, y_train):.4f}")
print(f"    Test accuracy: {tree_shallow.score(X_test, y_test):.4f}")

Decision trees can be used for feature selection by examining the feature importance scores.

In [ ]:
for idx in np.argsort(tree.feature_importances_)[::-1]:
    print(f"  {iris.feature_names[idx]}: {tree.feature_importances_[idx]:.4f}")

### Hyperparameter Tuning with GridSearchCV
We can use GridSearchCV to find the best hyperparameters for our decision tree classifier.

In [ ]:
from sklearn.model_selection import GridSearchCV
param_grid = {'max_depth': [2, 3, 4, 5, 10], 'min_samples_leaf': [1, 2, 4]}
grid_search = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5)
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

## Key Takeaways
* Decision trees are a fundamental concept in machine learning, used for both classification and regression tasks.
* Decision trees work by recursively splitting the data to create hierarchical decision rules, with each split maximizing information gain (reducing impurity).
* The depth of the tree and the number of leaves can give us insight into the complexity of the model.
* Decision trees can be sensitive to imbalanced data, where one class has a significantly larger number of instances than the others.
* Decision trees do not require feature scaling, as they do not use distance metrics.
* Decision trees are highly interpretable, as each path through the tree represents a decision rule.